# 강의 05 · 실습 2 — 평가 데이터셋과 채점자 · (5) 고난도 II


## 1. 문제상황

- 구름월드 안내 서비스의 후보는 이제 세 버전입니다. 버전 A는 두 문장 안에서 답만 합니다.
- 버전 B는 답 앞뒤에 감사 인사와 마무리 문장을 붙입니다. 버전 C는 답 뒤에 FAQ의 관련 항목을 한 문장 덧붙이고 마무리 문장도 붙입니다.
- 운영팀은 채점자 세 개(정답 부합·정중함·간결성)의 점수를 같은 무게로 더해 순위를 매기려 했습니다.
- 그런데 고객센터 팀장은 「정중함이 두 배로 중요하고, 답의 길이는 문제되지 않는다」고 말합니다.
- 같은 점수표라도 기준마다 무게를 다르게 주면 배포 후보가 달라질 수 있는데, 운영팀은 아직 두 가지 무게로 순위를 계산해 보지 않았습니다.


## 2. 문제와 목표

- **문제**: 후보가 셋이고 채점 기준이 셋인데, 기준마다 무게를 어떻게 주느냐에 따라 배포 후보가 달라질 수 있습니다. 무게 두 벌의 결과를 나란히 본 적이 없습니다.
- **목표**
  - 같은 골든 데이터셋으로 버전 A·B·C를 각각 평가합니다.
    - 세 버전: A는 답만, B는 감사 인사와 마무리 문장, C는 FAQ의 관련 항목 한 문장과 마무리 문장(단계 0의 서비스 코드에 있습니다)
    - 데이터셋 `sesac-lec05-ex02-weights`: 질문 6개(FAQ 안 4개 — 인사 1개 포함, 밖 2개)와 정답 텍스트의 짝. 값(`GOLDEN`)과 모델 채점자에 넣는 채점 지시문(`JUDGE_GUIDE`)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다
  - 채점자 세 개의 평균을 버전마다 모읍니다.
    - 정답 부합: 모델 채점자가 질문·정답·답을 읽고 0 또는 1점
    - 정중함: 답에 「말씀해 주세요」가 있으면 1.0, 없고 존댓말(「습니다」「세요」)이면 0.5, 둘 다 아니면 0
    - 간결성: 마침표·느낌표·물음표로 나눈 문장 수가 2 이하이면 1, 3 이상이면 0
  - 무게 두 벌로 가중 총점을 계산해 순위를 두 번 출력하고, 무게마다 배포 후보를 출력합니다.
    - 무게 두 벌: 같은 무게(세 기준 모두 1) / 정중함 2배·간결성 0(정답 부합 1, 정중함 2, 간결성 0)
    - 출력: 무게마다 순위 한 줄과 「배포 후보: 버전 X」 한 줄. 실험 이름 접두어는 `weights`
- **목표 달성 여부의 판정 기준**:
  - 버전 세 개의 채점자 평균이 표로 출력되고,
  - 무게마다 가중 총점의 순위가 출력되고 1위 버전이 배포 후보로 표시되며,
  - 두 무게의 배포 후보가 서로 다른 버전인 것을 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex02_s5_diagram.svg)


## 4. 단계별 요구사항

(이 단에서는 요구사항을 학생이 직접 씁니다. 「2. 문제와 목표」의 목표와 「3. 워크플로우 다이어그램」만 보고, 단계마다 무엇을 만들어야 하는지 번호 목록으로 적은 뒤 「6. 코드 — 스텝바이스텝」의 코드를 작성합니다.)


## 5. 코드 골격

(이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델을 준비합니다. `warnings.filterwarnings` 두 줄은 추적 라이브러리가 내는 직렬화 경고와 진행 막대 경고를 화면에서 감춥니다. 동작에는 영향이 없습니다. `re`는 답의 문장 수를 세는 규칙 채점자에서 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- `LANGSMITH_TRACING`을 켜고 프로젝트 이름을 `sesac-lec05-ex02`로 정하면, 아래에서 부르는 모델 호출과 실험이 LangSmith의 그 프로젝트에 남습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import io   # io — io.StringIO()에 client.evaluate가 출력하는 실험 URL 줄을 잡아 둡니다
import os
import re
import warnings
from contextlib import redirect_stdout   # redirect_stdout — with redirect_stdout(...): 안에서 client.evaluate를 불러 실험 URL 출력을 가립니다

from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langsmith import Client   # Client — client = Client()로 데이터셋 등록과 client.evaluate 실행에 씁니다
from pydantic import BaseModel, Field   # BaseModel·Field — 모델 채점자의 구조화 출력 스키마와 필드 설명(description=…)에 씁니다

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", message="IProgress not found")

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex02"

# 이 실습에서 부르는 모델 이름
MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
llm = init_chat_model(MODEL, model_provider="litellm")
print("모델 준비를 마쳤습니다. LangSmith 프로젝트:", os.environ["LANGSMITH_PROJECT"])

# 주어진 자료: 골든 데이터셋 GOLDEN(질문·정답 짝)과 모델 채점자에 넣는 지시문 JUDGE_GUIDE — 값을 그대로 씁니다
GOLDEN = [
    {"question": "안녕하세요!",
     "answer": "안녕하세요! 무엇을 도와드릴까요?", "must_know": True},
    {"question": "자유이용권 환불 규정 알려 주세요",
     "answer": "이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.", "must_know": True},
    {"question": "야간개장 때 퍼레이드 하나요?",
     "answer": "야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.", "must_know": True},
    {"question": "주차 요금은 얼마인가요?",
     "answer": "자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.", "must_know": True},
    {"question": "근처에 맛있는 식당이 있나요?",
     "answer": "해당 내용은 확인할 수 없습니다.", "must_know": False},
    {"question": "파이썬 리스트 정렬은 어떻게 하나요?",
     "answer": "해당 내용은 확인할 수 없습니다.", "must_know": False},
]

JUDGE_GUIDE = (
    "너는 엄격한 채점자다. 평가 대상의 답이 정답(reference)과 부합하는지, 어긋난 지점이 없는지 판정하라. "
    "인사말이나 마무리 문장이 덧붙은 것, FAQ에 적힌 관련 안내가 덧붙은 것은 어긋남으로 보지 않는다. "
    "정답에 없는 사실을 지어냈거나 정답과 다른 내용을 말했으면 어긋남이다."
)


평가 대상이 될 안내 서비스 세 버전입니다. 세 버전 모두 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_service`가 지시문을 받아 서비스 함수를 만들고, 세 버전은 지시문만 다릅니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

BASE_GUIDE = (
    "너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
    "인사말에는 짧은 인사로 답한다. "
    "FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다."
)


def make_service(guide: str):
    """안내 서비스를 만든다: 지시문 하나를 받아 질문 -> 답 함수를 돌려준다."""
    system = guide + "\n=== FAQ ===\n" + FAQ

    def answer(question: str) -> str:
        res = llm.invoke([("system", system), ("human", question)])
        return res.content.strip()

    return answer


GUIDE_A = BASE_GUIDE
GUIDE_B = (
    BASE_GUIDE
    + " 답의 앞에 '문의해 주셔서 감사합니다.'를 붙이고, 답의 뒤에 '더 궁금한 점이 있으면 말씀해 주세요.'를 붙인다."
)
GUIDE_C = (
    BASE_GUIDE
    + " 답의 뒤에 FAQ에 적힌 관련 항목 하나를 한 문장으로 덧붙이고, 그 뒤에 '더 궁금한 점이 있으면 말씀해 주세요.'를 붙인다."
)

answer_a = make_service(GUIDE_A)
answer_b = make_service(GUIDE_B)
answer_c = make_service(GUIDE_C)

for name, svc in (("A", answer_a), ("B", answer_b), ("C", answer_c)):
    print(f"[{name}]", svc("야간개장 때 퍼레이드 하나요?"))

In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 단계 0의 서비스 출력에서 세 버전의 답 모양이 다릅니다. 버전 A는 두 문장 이하, 버전 B는 감사 인사로 시작해 마무리 문장으로 끝나고, 버전 C는 FAQ의 관련 항목 한 문장이 덧붙고 마무리 문장으로 끝납니다.
2. 버전별 채점자 평균 표에서 정중함 점수는 B와 C가 A보다 높고, 간결성 점수는 A가 가장 높습니다.
3. 무게마다 가중 총점과 순위가 한 줄씩 출력되고, 이어서 「배포 후보: 버전 …」 줄이 출력됩니다.
4. 두 무게의 배포 후보가 서로 다른 버전입니다. 정중함 2배·간결성 0의 무게에서는 버전 B가 1위입니다.

네 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.
